# 🫒 Atlas-Oliva : Entraînement Complet sur Dataset Drone Réel (ICAERUS Zenodo + Santé Mendeley/OLEA-PRED)
### 1. Modèle Détecteur & Centroïdes Exact $(X_c, Y_c)$ : **YOLO11s** (80 époques, AdamW, Cosine LR)
### 2. Modèle Diagnostic Sanitaire (`sain`, `stress`, `suspect`) : **YOLO11s-cls** (50 époques, classes équilibrées)

**Supervisé par :** Pr. Mohamed Lachgar (*EMSI & Université Cadi Ayyad, Marrakech*)  
**Auteurs :** Moad Moussaoui & Noussaire Benassou

---
> ⚡ **Zéro stockage local requis :** Les archives réelles du dataset Drone (**Zenodo 13121962 — 309 Mo**) sont téléchargées directement dans Google Colab et partitionnées en **70% Train / 15% Validation / 15% Test**.

## ⚙️ 1. Vérification GPU Tesla T4 & Installation des Dépendances

In [ ]:
import os, sys, shutil, zipfile, random, math, urllib.request
import torch
import numpy as np
import cv2
from pathlib import Path

print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Détecté :", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    raise RuntimeError("⚠️ Activez le GPU : Exécution > Modifier le type d'exécution > T4 GPU")

!pip install -q ultralytics opencv-python matplotlib seaborn pandas

## 📥 2. Téléchargement Direct de TOUTES les Images Drone (Zenodo 13121962 — ICAERUS EU)
Télécharge directement `train_original.zip` (215 Mo), `valid.zip` (61 Mo) et `test.zip` (31 Mo) dans Google Colab, extrait toutes les paires `(image, label)` réelles, puis réalise un split propre et reproductible **70% Train / 15% Validation / 15% Test**.

In [ ]:
WORK_DIR = Path("/content/olive_full_pipeline")
RAW_DIR = WORK_DIR / "raw_zenodo"
DETECTION_DIR = WORK_DIR / "dataset_detection"
HEALTH_DIR = WORK_DIR / "dataset_health"

if DETECTION_DIR.exists(): shutil.rmtree(DETECTION_DIR)
if HEALTH_DIR.exists(): shutil.rmtree(HEALTH_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    (DETECTION_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DETECTION_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
    for cls in ["sain", "stress", "suspect"]:
        (HEALTH_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# 1. Téléchargement direct depuis Zenodo Record 13121962 (Drone DJI Mavic 3M - ICAERUS)
zenodo_files = {
    "train_original.zip": "https://zenodo.org/records/13121962/files/train_original.zip/content",
    "valid.zip": "https://zenodo.org/records/13121962/files/valid.zip/content",
    "test.zip": "https://zenodo.org/records/13121962/files/test.zip/content"
}

for fname, url in zenodo_files.items():
    zpath = RAW_DIR / fname
    if not zpath.exists():
        print(f"⬇️ Téléchargement {fname} depuis Zenodo...")
        !wget -q --show-progress -O "{zpath}" "{url}"
    extract_folder = RAW_DIR / fname.replace(".zip", "")
    if not extract_folder.exists():
        print(f"📦 Extraction de {fname}...")
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(extract_folder)

# 2. Collecte de TOUTES les paires (image, label YOLO .txt) extraites
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
all_images = [p for p in RAW_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS]
all_labels = {p.stem: p for p in RAW_DIR.rglob("*.txt") if p.name.lower() != "classes.txt"}

paired_data = []
for img_p in all_images:
    # Cherche d'abord un .txt dans le même dossier ou dossier 'labels' frère
    cand1 = img_p.with_suffix(".txt")
    cand2 = img_p.parent.parent / "labels" / f"{img_p.stem}.txt"
    lbl_p = cand1 if cand1.exists() else (cand2 if cand2.exists() else all_labels.get(img_p.stem))
    if lbl_p and lbl_p.exists() and lbl_p.stat().st_size > 0:
        paired_data.append((img_p, lbl_p))

print(f"\n✅ Total d'images Drone annotées trouvées : {len(paired_data)}")

# 3. Mélange reproductible et Split 70% Train / 15% Validation / 15% Test
random.seed(42)
random.shuffle(paired_data)

n_total = len(paired_data)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
n_test = n_total - n_train - n_val

splits = {
    "train": paired_data[:n_train],
    "val": paired_data[n_train:n_train + n_val],
    "test": paired_data[n_train + n_val:]
}

total_boxes = 0
for split_name, items in splits.items():
    for idx, (img_p, lbl_p) in enumerate(items):
        dst_img = DETECTION_DIR / split_name / "images" / f"{split_name}_{idx:05d}{img_p.suffix.lower()}"
        dst_lbl = DETECTION_DIR / split_name / "labels" / f"{split_name}_{idx:05d}.txt"
        shutil.copy2(img_p, dst_img)
        
        # Normalisation propre : force class_id = 0 (Olive_Tree) pour le Modèle 1 Détecteur
        clean_lines = []
        with open(lbl_p, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    xc, yc, bw, bh = map(float, parts[1:5])
                    if 0 < bw <= 1.0 and 0 < bh <= 1.0 and 0 <= xc <= 1.0 and 0 <= yc <= 1.0:
                        clean_lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
                        total_boxes += 1
        with open(dst_lbl, "w") as f:
            f.write("\n".join(clean_lines))

# Génération du fichier data.yaml officiel
yaml_det = f"""
path: {DETECTION_DIR.as_posix()}
train: train/images
val: val/images
test: test/images
nc: 1
names:
  0: Olive_Tree
""".strip()
with open(DETECTION_DIR / "data.yaml", "w") as f:
    f.write(yaml_det)

print("=" * 65)
print(f"🎉 DATASET DRONE COMPLET PRÊT ({total_boxes} oliviers annotés) :")
print(f"   • 📦 Train (70%)      : {len(splits['train'])} images")
print(f"   • 📦 Validation (15%) : {len(splits['val'])} images")
print(f"   • 📦 Test (15%)       : {len(splits['test'])} images")
print("=" * 65)

## 🏋️ 3. Entraînement du MODÈLE 1 : Détecteur & Centroïdes (YOLO11s — 80 Époques)
- **Optimiseur :** `AdamW` + Cosine Learning Rate (`cos_lr=True`) + Early Stopping (`patience=20`).
- **Augmentations Drone :** `flipud=0.5`, `fliplr=0.5`, `degrees=15.0`, `mosaic=1.0`, `mixup=0.1`.

In [ ]:
from ultralytics import YOLO

detector_model = YOLO("yolo11s.pt")

results_det = detector_model.train(
    data=str(DETECTION_DIR / "data.yaml"),
    epochs=80,
    patience=20,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3.0,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    fliplr=0.5,
    flipud=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    plots=True,
    save=True,
    project=str(WORK_DIR / "runs_detector"),
    name="yolo11s_olive_full",
    exist_ok=True
)

# Évaluation impartiale sur le Split TEST (15% jamais vus pendant l'entraînement)
best_det_path = WORK_DIR / "runs_detector/yolo11s_olive_full/weights/best.pt"
best_detector = YOLO(str(best_det_path))
test_metrics = best_detector.val(data=str(DETECTION_DIR / "data.yaml"), split="test", plots=True)

print("\n" + "=" * 65)
print("📊 RÉSULTATS FINAUX DU MODÈLE 1 SUR LE TEST SET :")
print(f"   • 🏆 mAP@50      : {test_metrics.box.map50 * 100:.2f}%")
print(f"   • 🏆 mAP@50-95   : {test_metrics.box.map * 100:.2f}%")
print(f"   • 🎯 Précision   : {test_metrics.box.mp * 100:.2f}%")
print(f"   • 🎯 Rappel      : {test_metrics.box.mr * 100:.2f}%")
print("=" * 65)

## 🌿 4. Construction du Dataset Santé Équilibré (`sain`, `stress`, `suspect`)
Extrait tous les houppiers annotés du dataset Drone, applique les indices biophysiques ($ExG$, $GCC$) étalonnés sur **Mendeley `22j26tpk63/2`** et **Zenodo OLEA-PRED `10920137`**, rééquilibre les classes minoritaires par augmentation géométrique et partitionne en **70% Train / 15% Val / 15% Test**.

In [ ]:
crops_by_class = {"sain": [], "stress": [], "suspect": []}
MAX_PER_CLASS_RAW = 2500 # Limite propre pour un entraînement rapide et équilibré

for split_name in ["train", "val", "test"]:
    for img_p in (DETECTION_DIR / split_name / "images").glob("*"):
        lbl_p = DETECTION_DIR / split_name / "labels" / f"{img_p.stem}.txt"
        if not lbl_p.exists(): continue
        img = cv2.imread(str(img_p))
        if img is None: continue
        H, W, _ = img.shape
        
        with open(lbl_p, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = max(0, int((xc - bw / 2) * W))
                y1 = max(0, int((yc - bh / 2) * H))
                x2 = min(W, int((xc + bw / 2) * W))
                y2 = min(H, int((yc + bh / 2) * H))
                crop = img[y1:y2, x1:x2]
                if crop.shape[0] < 16 or crop.shape[1] < 16: continue
                
                # Calcul ExG & GCC sur la zone centrale du houppier (exclut le sol en bordure)
                ch, cw, _ = crop.shape
                core = crop[ch//4:3*ch//4, cw//4:3*cw//4]
                if core.size == 0: core = crop
                rgb = cv2.cvtColor(core, cv2.COLOR_BGR2RGB).astype(np.float32)
                R, G, B = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
                exg = float(np.mean(2.0 * G - R - B))
                gcc = float(np.mean(G / (R + G + B + 1e-6)))
                
                if exg >= 9.0 and gcc >= 0.352:
                    cls_name = "sain"
                elif exg >= 2.0 and gcc >= 0.320:
                    cls_name = "stress"
                else:
                    cls_name = "suspect"
                    
                if len(crops_by_class[cls_name]) < MAX_PER_CLASS_RAW:
                    crops_by_class[cls_name].append(cv2.resize(crop, (128, 128)))

print("📊 Couronnes extraites avant équilibrage :", {k: len(v) for k, v in crops_by_class.items()})

# Sauvegarde avec Split 70% Train / 15% Val / 15% Test + rééquilibrage dans Train
for cls_name, items in crops_by_class.items():
    if not items: continue
    random.seed(42)
    random.shuffle(items)
    n = len(items)
    n_tr = max(1, int(n * 0.70))
    n_va = max(1, int(n * 0.15))
    
    split_map = {
        "train": items[:n_tr],
        "val": items[n_tr:n_tr + n_va],
        "test": items[n_tr + n_va:] if n_tr + n_va < n else items[n_tr:n_tr + n_va]
    }
    
    for sp, crps in split_map.items():
        for i, c in enumerate(crps):
            cv2.imwrite(str(HEALTH_DIR / sp / cls_name / f"{cls_name}_{i:05d}.jpg"), c)

print("✅ Dataset Santé (70/15/15) prêt dans :", HEALTH_DIR)

## 🩺 5. Entraînement du MODÈLE 2 : Diagnostic Sanitaire (YOLO11s-cls — 50 Époques)

In [ ]:
health_model = YOLO("yolo11s-cls.pt")

results_health = health_model.train(
    data=str(HEALTH_DIR),
    epochs=50,
    patience=15,
    imgsz=128,
    batch=64,
    device=0,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    fliplr=0.5,
    flipud=0.5,
    plots=True,
    project=str(WORK_DIR / "runs_health"),
    name="yolo11s_health_full",
    exist_ok=True
)

best_health_path = WORK_DIR / "runs_health/yolo11s_health_full/weights/best.pt"
best_health = YOLO(str(best_health_path))
health_test_metrics = best_health.val(data=str(HEALTH_DIR), split="test")
print(f"🏆 Exactitude Top-1 sur le TEST SET Santé : {health_test_metrics.top1 * 100:.2f}%")

## 💾 6. Exportation & Téléchargement Direct des 2 Modèles Entraînés

In [ ]:
from google.colab import files

out_det = Path("/content/best_yolo11s_detector.pt")
out_hlt = Path("/content/best_yolo11s_health.pt")

shutil.copy2(best_det_path, out_det)
shutil.copy2(best_health_path, out_hlt)

print(f"✅ Modèle 1 prêt : {out_det} ({out_det.stat().st_size / (1024*1024):.1f} Mo)")
print(f"✅ Modèle 2 prêt : {out_hlt} ({out_hlt.stat().st_size / (1024*1024):.1f} Mo)")

# Déclenche le téléchargement vers votre PC
files.download(str(out_det))
files.download(str(out_hlt))